### Celda 1 — Librerías y configuración de rutas

En esta celda se importan las librerías necesarias para:
- Carga y manejo de datos (`pandas`, `numpy`)
- Separación de datos en entrenamiento/validación/prueba (`train_test_split`)
- Evaluación del clasificador (matriz de confusión, reporte de clasificación y ROC AUC)
- Construcción del pipeline de preprocesamiento + modelo (escalado + regresión logística)
- Guardado del modelo entrenado para uso posterior (`joblib`)

Además, se definen las rutas:
- Carpeta del PASO 2 (salidas del entrenamiento)
- Archivo de entrada con features extraídas (`features_dataset_v1.csv`)
- Archivos de salida: modelo y reporte de resultados

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import joblib


#Dirección donde se generarán los archivos del modelo
BASE_DIR = r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2"

# Direccion de donde se tomará el dataset base generadp en PASO 1
FEATURES_CSV = r"C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\features_dataset_v1.csv"

OUT_MODEL  = os.path.join(BASE_DIR, "REGRESIONLOGISTICA_model.joblib")
OUT_REPORT = os.path.join(BASE_DIR, "REGRESIONLOGISTICA_report.txt")

print("BASE_DIR:", BASE_DIR)
print("FEATURES_CSV:", FEATURES_CSV)

BASE_DIR: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2
FEATURES_CSV: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 1\features_dataset_v1.csv


### Celda 2 — Carga del dataset de features y limpieza mínima

En esta celda:
1. Se carga el archivo `features_dataset_v1.csv` generado en el PASO 1.
2. Se aplica un *capping* (limitación) a `loudness_var_proxy_db` para reducir el efecto de outliers extremos
   producidos por segmentos con energía muy baja (artefactos numéricos).
   - Se limita el valor máximo a 40 dB, evitando que un único valor anómalo influya desproporcionadamente en el modelo.
3. Se define la variable objetivo binaria `y_bin`:
   - `0` = OK
   - `1` = NO_OK (advertencia o falla)
4. Se definen las columnas de entrada `X` excluyendo metadatos que no deben entrar como features
   (por ejemplo: nombre de archivo, tiempos, etc.).

Finalmente, se imprime el número de filas, el listado de features y la distribución de clases.

In [2]:
df = pd.read_csv(FEATURES_CSV)

# Capear outliers (como acordamos)
if "loudness_var_proxy_db" in df.columns:
    df["loudness_var_proxy_db"] = df["loudness_var_proxy_db"].clip(upper=40)

# Target
if "y_bin" not in df.columns:
    raise ValueError("No existe y_bin en el archivo features_dataset. Revisa Paso 1.")

y = df["y_bin"].astype(int)

# Columns que NO entran como features
non_feature_cols = ["clip_id", "file_name", "start_time_s", "end_time_s", "qc_result", "y_bin"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]

X = df[feature_cols].copy()

print("Filas:", len(df))
print("Features:", feature_cols)
print("Distribución y_bin:\n", y.value_counts())

Filas: 764
Features: ['rms_mean', 'crest_factor_db', 'true_peak_dbfs', 'short_term_lufs_mean', 'loudness_var_proxy_db', 'silence_ratio', 'stereo_correlation', 'spectral_centroid_mean', 'spectral_flatness_mean', 'clipping_ratio', 'near_ceiling_ratio']
Distribución y_bin:
 y_bin
0    399
1    365
Name: count, dtype: int64


### Celda 3 — División del dataset en entrenamiento, validación y prueba

Se divide el dataset en tres subconjuntos para evaluación objetiva:

- **Train (70%)**: usado para ajustar los parámetros del modelo.
- **Validation (15%)**: usado para revisar desempeño durante el desarrollo (sin tocar el test).
- **Test (15%)**: usado solo al final para reportar resultados finales del baseline.

Se utiliza `stratify=y` para asegurar que las proporciones de clases (OK/NO_OK)
se mantengan similares en los tres subconjuntos.

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

Train: 534 Val: 115 Test: 115


### Celda 4 — Entrenamiento del modelo baseline (Regresión Logística)

Se implementa un pipeline compuesto por:

1. **StandardScaler**:
   - Estandariza cada feature a media 0 y desviación estándar 1.
   - Es importante en regresión logística porque el modelo es sensible a escalas distintas.

2. **LogisticRegression**:
   - Se configura como baseline por su interpretabilidad y robustez.
   - Regularización **L2** para evitar sobreajuste.
   - `class_weight="balanced"` para compensar posibles diferencias en frecuencia de clases.
   - `solver="lbfgs"` y `max_iter=2000` para asegurar convergencia.

Finalmente, se entrena el pipeline con el conjunto de entrenamiento.

In [4]:
logreg = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        class_weight="balanced",
        solver="lbfgs",
        max_iter=2000,
        random_state=42
    ))
])

logreg.fit(X_train, y_train)
print("Entrenamiento listo.")

Entrenamiento listo.


c:\Users\Protools\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


### Celda 5 — Evaluación del modelo (Validación y Test)

Se evalúa el modelo en los subconjuntos de validación y prueba con:

- **Matriz de confusión**: muestra aciertos y errores por clase (OK vs NO_OK).
- **Classification report**: precisión, recall y F1 por clase.
- **ROC AUC**: capacidad del modelo para separar clases usando probabilidades.

Nota: En un sistema de control de calidad (QC), es especialmente importante el **recall de NO_OK**,
ya que se busca minimizar fallas no detectadas (falsos negativos).

In [5]:
def eval_split(name, model, X_split, y_split):
    pred = model.predict(X_split)
    proba = model.predict_proba(X_split)[:, 1]

    cm = confusion_matrix(y_split, pred)
    rep = classification_report(y_split, pred, digits=4)
    auc = roc_auc_score(y_split, proba)

    print(f"\n=== {name} ===")
    print("Confusion Matrix:\n", cm)
    print("\nClassification Report:\n", rep)
    print("ROC AUC:", round(auc, 4))

    return cm, rep, auc

cm_val, rep_val, auc_val = eval_split("VALIDACIÓN", logreg, X_val, y_val)
cm_test, rep_test, auc_test = eval_split("TEST", logreg, X_test, y_test)


=== VALIDACIÓN ===
Confusion Matrix:
 [[60  0]
 [ 3 52]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9524    1.0000    0.9756        60
           1     1.0000    0.9455    0.9720        55

    accuracy                         0.9739       115
   macro avg     0.9762    0.9727    0.9738       115
weighted avg     0.9752    0.9739    0.9739       115

ROC AUC: 0.9915

=== TEST ===
Confusion Matrix:
 [[58  2]
 [ 1 54]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9831    0.9667    0.9748        60
           1     0.9643    0.9818    0.9730        55

    accuracy                         0.9739       115
   macro avg     0.9737    0.9742    0.9739       115
weighted avg     0.9741    0.9739    0.9739       115

ROC AUC: 0.9979


### Celda 6 — Persistencia del modelo y reporte de resultados

Se guarda:
- El modelo entrenado completo (pipeline con escalado + clasificador).
- La lista de `feature_cols` para asegurar consistencia al predecir en producción.

Además, se genera un archivo de texto con el resumen de resultados (validación y test)
para documentación formal en la tesis.

In [6]:
os.makedirs(BASE_DIR, exist_ok=True)

# Guardar modelo
joblib.dump({
    "model": logreg,
    "feature_cols": feature_cols
}, OUT_MODEL)

# Guardar reporte
with open(OUT_REPORT, "w", encoding="utf-8") as f:
    f.write("BASELINE - REGRESIÓN LOGÍSTICA (BINARIO OK vs NO_OK)\n\n")
    f.write("CONFIG:\n")
    f.write("- penalty=L2\n- class_weight=balanced\n- scaler=StandardScaler\n\n")

    f.write("FEATURES:\n")
    f.write(", ".join(feature_cols) + "\n\n")

    f.write("=== VALIDACIÓN ===\n")
    f.write(rep_val + "\n")
    f.write(f"ROC AUC: {auc_val}\n\n")

    f.write("=== TEST ===\n")
    f.write(rep_test + "\n")
    f.write(f"ROC AUC: {auc_test}\n\n")

print("Modelo guardado en:", OUT_MODEL)
print("Reporte guardado en:", OUT_REPORT)

Modelo guardado en: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2\REGRESIONLOGISTICA_model.joblib
Reporte guardado en: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2\REGRESIONLOGISTICA_report.txt


### Celda 7 — Interpretación del modelo mediante coeficientes

Se extraen los coeficientes de la regresión logística para analizar el peso de cada feature.

Interpretación:
- Coeficientes **positivos** aumentan la probabilidad de clasificar como **NO_OK**.
- Coeficientes **negativos** disminuyen dicha probabilidad.
- La magnitud (valor absoluto) indica qué variables influyen más.

Esto permite justificar en la tesis qué características acústicas son más relevantes
para detectar problemas técnicos (por ejemplo: energía RMS, correlación estéreo y ratio de silencio).

In [7]:
# Extraer coeficientes de la regresión
clf = logreg.named_steps["clf"]
scaler = logreg.named_steps["scaler"]

coef = pd.Series(clf.coef_[0], index=feature_cols).sort_values(key=np.abs, ascending=False)
coef

coef_path = os.path.join(BASE_DIR, "logreg_coefficients.csv")
coef.to_csv(coef_path, header=["coef"])
print("Coeficientes guardados en:", coef_path)

Coeficientes guardados en: C:\Users\Protools\Desktop\TESIS\DESARROLLO TESIS\PASO 2\logreg_coefficients.csv


## Conclusión

Se entrenó y evaluó un modelo baseline de clasificación binaria (OK vs NO_OK) usando Regresión Logística
con regularización L2 y balanceo de clases. El modelo fue entrenado sobre un conjunto de entrenamiento
y evaluado en validación y prueba, obteniendo métricas altas de discriminación (ROC AUC)
y buen desempeño para la clase NO_OK, lo cual lo hace adecuado como primer filtro automático
para el sistema de control de calidad de audio.

El modelo y la configuración de features quedaron almacenados para su uso posterior
en la etapa de generación de reportes por archivo (PASO 3).